# micrograd, from scratch

This is the *Practice* step of `unit_01_micrograd.md`. Do the Cold Attempt there first.

Work top to bottom. Each milestone is one cell of stubs followed by a grader cell.
The grader stops at your first failure so there is always exactly one thing in front of you.

**Rules of engagement**
- Don't open the lecture. Don't open the real micrograd repo.
- Stuck on an *idea* for 20 min → ask the coaching chat for a hint.
- Stuck on *Python syntax* → ask immediately, zero learning value in that.
- **Before you run a grader cell, say out loud what you expect to happen.**

Later milestones add methods to `Value` with `Value.name = name`. That's just so you
can build the class up one cell at a time instead of re-running one giant cell.

In [1]:
import math
import random
from test_micrograd import grade

## Milestone 1 & 2 — a number that remembers, and one hop of gradient

Each op does two jobs: compute the forward number, and attach a closure that knows the
local derivative. Write both in the same method.

The container is given. The ideas are the two methods below it.

In [2]:
class Value:
    """A scalar that remembers the operation that produced it.

    Fields:
      data       float, the actual number
      grad       float, d(final output) / d(self). starts at zero.
      _prev      set of Values that were the inputs to the op producing self
      _op        str, debug label for that op
      _backward  a closure that takes self.grad and pushes gradient ONE HOP
                 back into each element of _prev. does nothing for a leaf.
    """

    def __init__(self, data, _children=(), _op=''):
        self.data = data
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op

    def __repr__(self):
        return f"Value(data={self.data}, grad={self.grad})"

    def dump(self, indent=0):
        """Plumbing: print the expression graph as text. For debugging."""
        pad = '  ' * indent
        label = self._op or 'leaf'
        print(f"{pad}{label:>6} data={self.data:<12.6g} grad={self.grad:<12.6g}")
        for child in self._prev:
            child.dump(indent + 1)

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), "+")

        def backward():
            # addition passes gradient through untouched since derivative is 1
            self.grad += out.grad * 1
            other.grad += out.grad * 1
        out._backward = backward

        return out

    def __mul__(self, other):
        # in here for c = a * b, self would be a, b would be other, and c is out
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), "*")
        def backward():
            self.grad += out.grad * other.data
            other.grad += out.grad * self.data
        out._backward = backward
        return out

In [3]:
a, b = Value(2.0), Value(-3.0)
c = a*b

# dL/dL = 1.0
c.grad = 1.0
c._backward()

c.dump()

     * data=-6           grad=1           
    leaf data=-3           grad=2           
    leaf data=2            grad=-3          


In [4]:
grade(Value, upto=2)

[ok]   milestone 1: forward pass + graph bookkeeping (__add__, __mul__)
[ok]   milestone 2: one hop of gradient (_backward closures)

all milestones passed.


0

## Milestone 3 & 4 — the whole graph, in the right order

Two questions to answer before you write a line:
1. In what order must the nodes be visited, and why that order?
2. What is `self.grad`, before any of this starts?

Milestone 4 has no new stub. It's a one-character change somewhere above, and the grader will tell you when you need it.

1. reverse topological, so every consumer of x has its gradient computed first (going backwards from the output)
2. self.grad at the start is 1.0 b/c dLoss/dLoss is 1

In [5]:
def backward(self):
    """Run backprop from self through the entire graph behind it."""
    # copied reverse topological sort
    topo = []
    visited = set()
    def build_topo(v):
        if v not in visited:
            visited.add(v)
            for child in v._prev:
                build_topo(child)
            topo.append(v)
    build_topo(self)

    self.grad = 1.0
    for node in reversed(topo):
        # call local grad
        node._backward()

Value.backward = backward

In [6]:
a, b = Value(2.0), Value(-3.0)
c = a*b

# dL/dL = 1.0
c.grad = 1.0
c.backward()

c.dump()

     * data=-6           grad=1           
    leaf data=2            grad=-3          
    leaf data=-3           grad=2           


In [7]:
grade(Value, upto=4)

[ok]   milestone 1: forward pass + graph bookkeeping (__add__, __mul__)
[ok]   milestone 2: one hop of gradient (_backward closures)
[ok]   milestone 3: full backward() in the right order
[ok]   milestone 4: gradient accumulation on reused nodes

all milestones passed.


0

## Milestone 5 — tanh

You get to treat this as a single atomic op with one local derivative, even though
it is really exp/div/sub underneath.

In [8]:
def tanh(self):
    out = Value(math.tanh(self.data), (self, ), "tanh")

    def _backward():
        self.grad += out.grad * (1-out.data**2)
    out._backward = _backward
    return out

Value.tanh = tanh

In [9]:
grade(Value, upto=5)

[ok]   milestone 1: forward pass + graph bookkeeping (__add__, __mul__)
[ok]   milestone 2: one hop of gradient (_backward closures)
[ok]   milestone 3: full backward() in the right order
[ok]   milestone 4: gradient accumulation on reused nodes
[ok]   milestone 5: tanh

all milestones passed.


0

## Milestone 6 (stretch) — granularity

`exp` and `pow` need real `_backward` closures. Everything after them does **not**:
each is a one-liner built out of ops you already have. If you find yourself writing a
closure for `__sub__`, stop and think.

The four `__r*__` methods fire when the Value is on the *right* of the operator,
e.g. `2.0 * v` or `2.0 - v`. Pure Python trivia, ask if annoying.

In [10]:
def exp(self):
    raise NotImplementedError

def __pow__(self, other):
    assert isinstance(other, (int, float)), "only scalar exponents"
    raise NotImplementedError

def __neg__(self):
    raise NotImplementedError

def __sub__(self, other):
    raise NotImplementedError

def __truediv__(self, other):
    raise NotImplementedError

def __radd__(self, other):
    raise NotImplementedError

def __rmul__(self, other):
    raise NotImplementedError

def __rsub__(self, other):
    raise NotImplementedError

def __rtruediv__(self, other):
    raise NotImplementedError

for _f in (exp, __pow__, __neg__, __sub__, __truediv__,
           __radd__, __rmul__, __rsub__, __rtruediv__):
    setattr(Value, _f.__name__, _f)

In [11]:
grade(Value, upto=6)

[ok]   milestone 1: forward pass + graph bookkeeping (__add__, __mul__)
[ok]   milestone 2: one hop of gradient (_backward closures)
[ok]   milestone 3: full backward() in the right order
[ok]   milestone 4: gradient accumulation on reused nodes
[ok]   milestone 5: tanh

[TODO] milestone 6: STRETCH: exp, pow, neg, sub, div, r-ops
    exp raised NotImplementedError -- this is the next thing to write.



1

## Milestone 7 (stretch) — it learns

Nothing below here knows anything about calculus. That is the point: once `Value`
works, a neural net is just arithmetic on top of it.

The grader stops at the first failure, in this order: Neuron param count → Neuron
output → Neuron gradients → Layer → MLP → training. So build one class, run the
grader, fix, repeat.

**Neuron(nin)**
- Parameters: `nin` weights plus one bias. `Neuron(3)` has 4 parameters. Each is a
  `Value(random.uniform(-1, 1))`.
- `__call__(x)`: the squashed weighted sum, `tanh(w1*x1 + w2*x2 + ... + b)`, computed
  with Value arithmetic so the weights are *in the graph*. Multiplying `w.data`
  gives the right number but nothing reaches the weights in backward.
- `parameters()`: list of the weights plus the bias.

**Layer(nin, nout)**
- `nout` neurons, each a `Neuron(nin)`.
- `__call__(x)`: run every neuron on the same `x`. Return the list, or the bare
  Value when `nout == 1`.
- `parameters()`: every neuron's parameters, flattened into one list.

**MLP(nin, nouts)**
- `MLP(3, [4, 4, 1])` is `Layer(3,4)`, `Layer(4,4)`, `Layer(4,1)`. Sizes chain:
  `[nin] + nouts`, consecutive pairs.
- `__call__(x)`: feed `x` through each layer in turn, output of one is input to the next.
- `parameters()`: every layer's parameters, flattened.


In [12]:
# required for M7
def __pow__(self, other):
    assert isinstance(other, (int, float)), "only scalar exponents"
    out = Value(self.data ** other, (self,), f"**{other}")
    def _backward():
        self.grad += other * self.data ** (other - 1) * out.grad
    out._backward = _backward
    return out

def __sub__(self, other):
    return self + (-1 * other)   # no closure needed, built from + and *

def __radd__(self, other):      # fires for 0 + v, e.g. inside sum()
    return self + other

def __rmul__(self, other):      # fires for -1 * v
    return self * other

for _f in (__pow__, __sub__, __radd__, __rmul__):
    setattr(Value, _f.__name__, _f)


In [ ]:
class Neuron:
    def __init__(self, nin):
        """nin weights and one bias. init weights uniform in [-1, 1]."""
        self.params = [Value(random.uniform(-1, 1)) for _ in range(nin)]
        self.bias = Value(random.uniform(-1, 1))

    def __call__(self, x):
        """x is a list of nin floats or Values. Return one Value: the
        squashed weighted sum."""
        # (x1w1 + x2w2 ..) + bias
        raw_sum = 0
        for xi, wi in zip(x, self.params):
            raw_sum += (xi * wi)
        activation = raw_sum + self.bias
        return activation.tanh()


    def parameters(self):
        """Flat list of every Value the optimizer is allowed to nudge."""
        return list(self.params) + [self.bias]

class Layer:
    def __init__(self, nin, nout):
        self.neurons = [Neuron(nin) for _ in range(nout)]

    def __call__(self, x):
        """Return a list of Values -- or a bare Value if nout == 1."""
        if len(self.neurons) == 1:
            return self.neurons[0](x)
        # feed X to every neuron in this layer and have a list of values
        activations = []
        for neuron in self.neurons:
            activations.append(neuron(x))
        return activations


    def parameters(self):
        params = []
        for n in self.neurons:
            params.extend(n.parameters())
        return params

# **MLP(nin, nouts)**
# - `__call__(x)`: feed `x` through each layer in turn, output of one is input to the next.
# - `parameters()`: every layer's parameters, flattened.
class MLP:
    def __init__(self, nin, nouts):
        """nouts is a list of layer widths, e.g. MLP(3, [4, 4, 1])."""
        sizes = [nin] + nouts
        self.layers = [Layer(sizes[i], sizes[i + 1]) for i in range(len(nouts))]

    def __call__(self, x):
        # feed x to first layer, then that layer's output to each following layer
        for layer in self.layers:
            x = layer(x)
        return x


    def parameters(self):
        params = []
        for l in self.layers:
            params.extend(l.parameters())
        return params

In [36]:
grade(Value, Neuron, Layer, MLP, skip=(1,2,3,4,5,6,))


[skip] milestone 1: forward pass + graph bookkeeping (__add__, __mul__)
[skip] milestone 2: one hop of gradient (_backward closures)
[skip] milestone 3: full backward() in the right order
[skip] milestone 4: gradient accumulation on reused nodes
[skip] milestone 5: tanh
[skip] milestone 6: STRETCH: exp, pow, neg, sub, div, r-ops
[ok]   milestone 7: STRETCH: Neuron / Layer / MLP, and it learns

all milestones passed.


0

## Your own training loop

The grader ran a training loop for you in milestone 7. Now write one yourself, from
memory, on the same toy data. Order of operations matters; if the loss does something weird, ask.

In [ ]:
xs = [[2.0, 3.0, -1.0], [3.0, -1.0, 0.5], [0.5, 1.0, 1.0], [1.0, 1.0, -1.0]]
ys = [1.0, -1.0, -1.0, 1.0]
model = MLP(3, [4, 4, 1])

for step in range(50):
    # forward
    pred = MLP(xs)



    # backward
    ## zero gradients
    ## compute loss
    ## back prop


    # update

NotImplementedError: 